# Intermezzo · Piccolo cabotaggio

I pesi e la busta, il viaggio, il venditore: i conti per capire se un modello sta nel tuo gozzo, e come andrà.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def conta(nome, valore, atteso):
    """Come controlla, per i numeri interi: devono coincidere esattamente."""
    ok = valore == atteso
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    intero = lambda x: f"{x:,.0f}".replace(",", ".")
    print(f"{segno}  {nome}: calcolato {intero(valore)}, nel libro {intero(atteso)}")
    assert ok, nome

## I pesi e la busta

*Nel racconto: il modello da 27 miliardi scritto a 4 bit; la busta che arriva a pesare quanto i pesi; lo spazio prenotato per tutta la pagina.*

Memoria dei pesi = parametri × bit ÷ 8. La busta si conta come per la nave di linea: byte per token = 2 × strati × teste KV × dimensione della testa × byte per numero.

In [2]:
def gb_pesi(parametri, bit):
    return parametri * bit / 8 / 1e9

def byte_per_token(strati, teste_kv, dim_testa, byte=2):
    return 2 * strati * teste_kv * dim_testa * byte

controlla("pesi del 27 miliardi a 4 bit (GB)", gb_pesi(27e9, 4), 13.5, 0)
pesi_gb = 15                                     # circa 15 con le scale dei gruppi

bpt = byte_per_token(48, 8, 128)                 # 48 strati, 8 teste KV di dimensione 128, a 16 bit
conta("byte per token", bpt, 196_608)
controlla("megabyte per token", bpt / 1e6, 0.2, 0.005)

def gb_busta(token, byte=2):
    return byte_per_token(48, 8, 128, byte) * token / 1e9

controlla("busta a 32.000 token (GB)", gb_busta(32_000), 6.3, 0.05)
controlla("busta a 128.000 token (GB)", gb_busta(128_000), 25, 0.2)
controlla("scritta a 8 bit, in rapporto a quella a 16", gb_busta(32_000, byte=1) / gb_busta(32_000), 0.5, 0)
assert gb_busta(128_000) > pesi_gb               # a pagina piena la busta pesa più dei pesi
print(f"a 128.000 token: {gb_busta(128_000):.0f} GB di busta contro {pesi_gb} GB di pesi")

✓  pesi del 27 miliardi a 4 bit (GB): calcolato 13.5, nel libro 13.5
✓  byte per token: calcolato 196.608, nel libro 196.608
✓  megabyte per token: calcolato 0.1966, nel libro 0.2
✓  busta a 32.000 token (GB): calcolato 6.291, nel libro 6.3
✓  busta a 128.000 token (GB): calcolato 25.17, nel libro 25
✓  scritta a 8 bit, in rapporto a quella a 16: calcolato 0.5, nel libro 0.5
a 128.000 token: 25 GB di busta contro 15 GB di pesi


## Il viaggio

*Nel racconto: la risposta che in fondo a una conversazione lunga scorre più lenta; il modello a esperti, sei o sette volte più svelto sulla carta; due o tre parole al secondo quando i pesi non stanno nella scheda.*

Token scritti al secondo, al massimo ≈ banda ÷ (byte dei pesi al lavoro + byte della busta). Con 15 GB di pesi e una banda di 400 GB/s.

In [3]:
def token_al_secondo(banda_gb, pesi_al_lavoro_gb, busta_gb):
    return banda_gb / (pesi_al_lavoro_gb + busta_gb)

banda = 400
denso = {}
for pagina, atteso in [(0, 27), (32_000, 19), (128_000, 10)]:
    denso[pagina] = token_al_secondo(banda, pesi_gb, gb_busta(pagina))
    pagina_scritta = f"{pagina:,}".replace(",", ".")
    controlla(f"denso, a {pagina_scritta} token di pagina", denso[pagina], atteso, 0.5)

print(f"a pagina piena va al {denso[128_000] / denso[0]:.0%} della pagina vuota")
assert denso[128_000] < denso[0] / 2             # «a meno della metà»

✓  denso, a 0 token di pagina: calcolato 26.67, nel libro 27
✓  denso, a 32.000 token di pagina: calcolato 18.79, nel libro 19
✓  denso, a 128.000 token di pagina: calcolato 9.959, nel libro 10
a pagina piena va al 37% della pagina vuota


Il modello a esperti, con 4 miliardi al lavoro per ogni token.

In [4]:
esperti_gb = gb_pesi(4e9, 4) * pesi_gb / 13.5    # 2 GB, più le scale dei gruppi come per i 27 miliardi
controlla("pesi al lavoro per token (GB)", esperti_gb, 2.2, 0.05)

a_vuoto = token_al_secondo(banda, 2.2, 0)
controlla("a esperti, a pagina vuota", a_vuoto, 180, 2)
print(f"a pagina vuota: {a_vuoto / denso[0]:.1f} volte il denso")
assert 6 <= a_vuoto / denso[0] <= 7              # «sei o sette volte»

a_32000 = token_al_secondo(banda, 2.2, gb_busta(32_000))
controlla("a esperti, a 32.000 token", a_32000, 47, 0.5)
controlla("a 32.000 token, volte il denso", a_32000 / denso[32_000], 2.5, 0.05)

✓  pesi al lavoro per token (GB): calcolato 2.222, nel libro 2.2
✓  a esperti, a pagina vuota: calcolato 181.8, nel libro 180
a pagina vuota: 6.8 volte il denso
✓  a esperti, a 32.000 token: calcolato 47.11, nel libro 47
✓  a 32.000 token, volte il denso: calcolato 2.507, nel libro 2.5


Quando una parte dei pesi resta fuori dalla scheda, conta la banda della memoria del computer: 20 GB fuori dalla scheda, a 60 GB/s.

In [5]:
controlla("secondi per ogni token", 20 / 60, 1 / 3, 1e-9)
controlla("token al secondo, al massimo", 60 / 20, 3, 0)

✓  secondi per ogni token: calcolato 0.3333, nel libro 0.333333
✓  token al secondo, al massimo: calcolato 3, nel libro 3


## Il venditore

*Nel racconto: la pagina di diecimila token, due o tre secondi sulla nave di linea, mezzo minuto o un minuto a casa; la prima parola che, con i reparti, arriva prima.*

Tempo alla prima parola ≈ 2 × parametri al lavoro × token della pagina ÷ operazioni al secondo.

In [6]:
def prima_parola(parametri_al_lavoro, token, operazioni_al_secondo):
    return 2 * parametri_al_lavoro * token / operazioni_al_secondo

conta("denso: operazioni per 10.000 token (miliardi)", 2 * 27e9 * 10_000 / 1e9, 540_000)
controlla("a ventimila miliardi al secondo, «una trentina» (secondi)", prima_parola(27e9, 10_000, 20_000e9), 30, 3)
controlla("a diecimila, «quasi un minuto» (secondi)", prima_parola(27e9, 10_000, 10_000e9), 60, 6)

conta("a esperti: operazioni per 10.000 token (miliardi)", 2 * 4e9 * 10_000 / 1e9, 80_000)
controlla("a esperti, a ventimila miliardi al secondo (secondi)", prima_parola(4e9, 10_000, 20_000e9), 4, 0)

# un acceleratore di punta: «qualche centinaio di migliaia di miliardi al secondo»
for migliaia in [300, 500]:
    s = prima_parola(27e9, 10_000, migliaia * 1000e9)
    print(f"acceleratore a {migliaia}.000 miliardi al secondo: {s:.1f} secondi")
    assert 1 <= s <= 2                           # «uno o due secondi»

✓  denso: operazioni per 10.000 token (miliardi): calcolato 540.000, nel libro 540.000
✓  a ventimila miliardi al secondo, «una trentina» (secondi): calcolato 27, nel libro 30
✓  a diecimila, «quasi un minuto» (secondi): calcolato 54, nel libro 60
✓  a esperti: operazioni per 10.000 token (miliardi): calcolato 80.000, nel libro 80.000
✓  a esperti, a ventimila miliardi al secondo (secondi): calcolato 4, nel libro 4
acceleratore a 300.000 miliardi al secondo: 1.8 secondi
acceleratore a 500.000 miliardi al secondo: 1.1 secondi


## I conti a margine, a casa

*Nel racconto: mille token di ragionamento a venti token al secondo; il documento in più nella busta.*

In [7]:
controlla("secondi prima che la risposta cominci", 1000 / 20, 50, 0)
controlla("megabyte di busta in più", 1000 * bpt / 1e6, 200, 5)

✓  secondi prima che la risposta cominci: calcolato 50, nel libro 50
✓  megabyte di busta in più: calcolato 196.6, nel libro 200


## Kata per casa · Esercizio 1

Un modello tipo da 8 miliardi, scritto a 4 bit, pesa 4,5 GB e ha 32 strati e 8 teste KV di dimensione 128. Quanto pesa la sua busta a 32.000 token, scritta a 16 bit e a 8 bit? Quanti token al secondo scrive, al massimo, con una banda di 100 GB/s, a pagina vuota e a 32.000 token?

In [8]:
bpt8 = byte_per_token(32, 8, 128)
conta("byte per token", bpt8, 131_072)
controlla("megabyte per token", bpt8 / 1e6, 0.13, 0.005)

b16 = bpt8 * 32_000 / 1e9
controlla("busta a 16 bit (GB)", b16, 4.2, 0.05)
controlla("busta a 8 bit (GB)", b16 / 2, 2.1, 0.05)

controlla("token al secondo a pagina vuota", token_al_secondo(100, 4.5, 0), 22, 0.5)
# il libro arrotonda prima i GB che viaggiano: 100 ÷ 8,7 e 100 ÷ 6,6
controlla("a 32.000 token, con la busta a 16 bit: GB che viaggiano", 4.5 + b16, 8.7, 0.05)
controlla("a 32.000 token, con la busta a 16 bit: 100 ÷ 8,7", 100 / 8.7, 11.5, 0.05)
controlla("a 32.000 token, con la busta a 8 bit: GB che viaggiano", 4.5 + b16 / 2, 6.6, 0.05)
controlla("a 32.000 token, con la busta a 8 bit: 100 ÷ 6,6", 100 / 6.6, 15, 0.5)
print(f"senza arrotondare a metà strada: {token_al_secondo(100, 4.5, b16):.1f} e {token_al_secondo(100, 4.5, b16 / 2):.1f} token al secondo")

✓  byte per token: calcolato 131.072, nel libro 131.072
✓  megabyte per token: calcolato 0.1311, nel libro 0.13
✓  busta a 16 bit (GB): calcolato 4.194, nel libro 4.2
✓  busta a 8 bit (GB): calcolato 2.097, nel libro 2.1
✓  token al secondo a pagina vuota: calcolato 22.22, nel libro 22
✓  a 32.000 token, con la busta a 16 bit: GB che viaggiano: calcolato 8.694, nel libro 8.7
✓  a 32.000 token, con la busta a 16 bit: 100 ÷ 8,7: calcolato 11.49, nel libro 11.5
✓  a 32.000 token, con la busta a 8 bit: GB che viaggiano: calcolato 6.597, nel libro 6.6
✓  a 32.000 token, con la busta a 8 bit: 100 ÷ 6,6: calcolato 15.15, nel libro 15
senza arrotondare a metà strada: 11.5 e 15.2 token al secondo


## Kata per casa · Esercizio 2

Il metodo: dalla memoria che il calcolo può usare togli qualche GB di margine e la busta per 32.000 token. Quello che resta, in GB, moltiplicato per 8 e diviso per i bit, dà i miliardi di parametri che ci stanno.

In [9]:
controlla("30 GB a 4 bit, miliardi di parametri", 30 * 8 / 4, 60, 0)

✓  30 GB a 4 bit, miliardi di parametri: calcolato 60, nel libro 60


Metti i numeri del tuo computer.

In [10]:
memoria_gb = 32                  # la memoria che il calcolo può usare
margine_gb = 2
mb_per_token = 0.2               # da 0,1 a 0,3 secondo il modello: lo dice il file di configurazione
bit = 4
operazioni_al_secondo = 20_000e9 # la potenza di calcolo del tuo computer

busta_gb = mb_per_token * 32_000 / 1000
miliardi = (memoria_gb - margine_gb - busta_gb) * 8 / bit
print(f"ci sta un modello fino a circa {miliardi:.0f} miliardi di parametri a {bit} bit")

attesa = 2 * miliardi * 1e9 * 10_000 / operazioni_al_secondo
print(f"prima parola su una pagina di 10.000 token: circa {attesa:.0f} secondi (con un modello a esperti contano solo i parametri al lavoro)")

ci sta un modello fino a circa 47 miliardi di parametri a 4 bit
prima parola su una pagina di 10.000 token: circa 47 secondi (con un modello a esperti contano solo i parametri al lavoro)
